# Restricted-input method reference
Source: Figure4_Park_Use_LQ_Maps.ipynb (workspace root)
Computes the Global Moran's I values reported in the manuscript Results (all park hexagons, n = 112/365/43; outcome = log1p of the per-hex sum of the HDB, non-landed and landed columns). It also draws the Figure 4 summary chart and location-quotient maps; the final figure layout was not verified. Requires the three private housing-stratified hexagon GeoJSONs under gps_analysis/general_model/duration_analysis/. Not an end-to-end pipeline. Figures and LISA maps show real cell-level patterns, so keep the working directory private. The Moran summary export cell was added for this release. See docs/MANUSCRIPT_ALIGNMENT.md.

In [ ]:
import os
from pathlib import Path
private_root = Path(os.environ["UFUG_PRIVATE_ROOT"]).resolve()
os.chdir(private_root)


In [ ]:
from pathlib import Path
import warnings; warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
import geopandas as gpd
import matplotlib as mpl
import matplotlib.pyplot as plt
mpl.rcParams["svg.fonttype"] = "none"
mpl.rcParams["svg.hashsalt"] = "park-use-lq"
import matplotlib.colors as mcolors
from matplotlib.patches import Patch
from IPython.display import display

# ---- file paths (edit to your locations) ----
FILES = {
    "amk":     "gps_analysis/general_model/duration_analysis/amk_duration_analysis_byhousingtypes.geojson",
    "jurong":  "gps_analysis/general_model/duration_analysis/jurong_duration_analysis_byhousingtypes.geojson",
    "punggol": "gps_analysis/general_model/duration_analysis/punggol_duration_analysis_byhousingtypes.geojson",
}

PARK_LABEL = {"amk": "Bishan-Ang Mo Kio Park", "jurong": "Jurong Lake Gardens", "punggol": "Punggol Waterway Park"}
PARK_TYPE  = {"amk": "neighbourhood", "jurong": "destination", "punggol": "linear corridor"}
PARK_ORDER = ["amk", "jurong", "punggol"]

# column order across the map (left -> right)
GROUPS      = ["hdb", "nolanded", "landed"]
GROUP_LABEL = {"hdb": "HDB", "nolanded": "Private (non-landed)", "landed": "Private (landed)"}

# metric key -> column prefix in your attribute table
PREFIX       = {"freq": "F_h_", "dur": "D_h_"}
METRIC_LABEL = {"freq": "visitation frequency", "dur": "dwell time"}

# projected CRS for Singapore (metres): needed for centroids, contiguity, scale bars
PROJ_CRS = 3414  # SVY21 / Singapore TM

# diverging colour scale on log2(LQ): -2..2 maps 0.25x .. 4x, centred on 1x
LQ_LOG2_MIN, LQ_LOG2_MAX = -2.0, 2.0
CMAP = "RdBu_r"            # red = over-used, blue = under-used (colour-blind safe)

# scale-bar length per park (metres) and a reliability floor (hexes below this
# TOTAL count are noisy for LQ; raise to grey out sparse cells, 0 = keep all)
SCALEBAR_M    = {"amk": 1000, "jurong": 500, "punggol": 500}
MIN_HEX_TOTAL = {"freq": 0, "dur": 0}

OUTDIR = Path("figures"); OUTDIR.mkdir(exist_ok=True)
print("config loaded; output ->", OUTDIR.resolve())

In [ ]:
def load_parks(files, proj_crs=PROJ_CRS):
    parks = {}
    needed = [f"{pre}{g}" for pre in PREFIX.values() for g in GROUPS]
    for key, path in files.items():
        gdf = gpd.read_file(path)
        missing = [c for c in needed if c not in gdf.columns]
        if missing:
            raise ValueError(f"{key}: missing columns {missing}")
        if gdf.crs is None:
            gdf = gdf.set_crs(4326)              # GeoJSON default is lon/lat
        gdf = gdf.to_crs(proj_crs)
        gdf["geometry"] = gdf.geometry.buffer(0)  # repair any invalid hexes
        parks[key] = gdf.reset_index(drop=True)
        print(f"{key:8s} {len(gdf):4d} hexes   CRS -> EPSG:{gdf.crs.to_epsg()}")
    return parks

parks = load_parks(FILES)

In [ ]:
def add_location_quotients(gdf):
    # adds <metric>_total and LQ_<metric>_<group> columns; LQ is within-park
    for m, pre in PREFIX.items():
        cols = [f"{pre}{g}" for g in GROUPS]
        V_h = gdf[cols].sum(axis=1)                 # per-hex total (row)
        V_g = gdf[cols].sum(axis=0)                 # per-group total (column)
        V   = float(gdf[cols].to_numpy().sum())     # grand total for this park
        gdf[f"{m}_total"] = V_h
        for g in GROUPS:
            v  = gdf[f"{pre}{g}"]
            lq = (v * V) / (V_h * V_g[f"{pre}{g}"])  # = (v/V_h)/(V_g/V)
            gdf[f"LQ_{m}_{g}"] = lq.where(V_h > 0, np.nan)
    return gdf

for k in parks:
    parks[k] = add_location_quotients(parks[k])

# peek
parks["amk"].filter(regex="grid_id|^freq_total$|^LQ_freq_").head()

In [ ]:
def summary_table(parks, over="all"):
    rows = []
    for k in PARK_ORDER:
        g = parks[k]; tot_f = g["freq_total"]; tot_d = g["dur_total"]
        vis = tot_f > 0
        sel = slice(None) if over == "all" else vis
        f, d = tot_f[sel], tot_d[sel]
        rows.append(dict(
            park=PARK_LABEL[k], n_hex=len(g),
            share_visited=round(float(vis.mean()), 3),
            mean_F=round(float(f.mean()), 2), median_F=round(float(np.median(f)), 1),
            mean_D=round(float(d.mean()), 0),
        ))
    return pd.DataFrame(rows)

print("Over ALL hexes in each file:")
display(summary_table(parks, "all"))
print("Over VISITED hexes only:")
display(summary_table(parks, "visited"))

In [ ]:
def lq_to_log2(series, lo=LQ_LOG2_MIN, hi=LQ_LOG2_MAX):
    # log2(LQ) clamped to [lo, hi]; LQ==0 -> lo (deep blue); NaN stays NaN (unvisited)
    s = series.astype(float).copy()
    s = s.mask(s == 0, 2.0 ** lo)            # avoid log2(0) = -inf
    return np.log2(s).clip(lo, hi)

def add_scalebar(ax, length_m=500, label=None, pad=0.06):
    x0, x1 = ax.get_xlim(); y0, y1 = ax.get_ylim()
    x = x0 + (x1 - x0) * pad; y = y0 + (y1 - y0) * pad
    ax.plot([x, x + length_m], [y, y], color="black", lw=2.4, solid_capstyle="butt", zorder=9)
    txt = label or (f"{length_m/1000:g} km" if length_m >= 1000 else f"{length_m:g} m")
    ax.text(x + length_m / 2, y + (y1 - y0) * 0.012, txt, ha="center", va="bottom", fontsize=7)

def add_north(ax):
    ax.annotate("N", xy=(0.07, 0.90), xytext=(0.07, 0.74),
                xycoords="axes fraction", textcoords="axes fraction",
                ha="center", va="center", fontsize=9, fontweight="bold",
                arrowprops=dict(arrowstyle="-|>", color="black", lw=1.3))

MISSING = dict(color="white", edgecolor="#cccccc", linewidth=0.3)  # unvisited style

def plot_lq_grid(parks, metric, overlays=None, lisa=None, save=None):
    norm = mcolors.Normalize(LQ_LOG2_MIN, LQ_LOG2_MAX)
    cmap = plt.get_cmap(CMAP)
    nr, nc = len(PARK_ORDER), len(GROUPS)
    fig, axes = plt.subplots(nr, nc, figsize=(4.0 * nc, 3.7 * nr))
    floor = MIN_HEX_TOTAL.get(metric, 0)
    for i, pk in enumerate(PARK_ORDER):
        g = parks[pk].copy()
        for j, grp in enumerate(GROUPS):
            ax = axes[i, j]
            val = lq_to_log2(g[f"LQ_{metric}_{grp}"])
            if floor > 0:                                  # optional reliability mask
                val = val.where(g[f"{metric}_total"] >= floor, np.nan)
            g["_v"] = val
            g.plot(column="_v", cmap=cmap, norm=norm, ax=ax,
                   edgecolor="white", linewidth=0.2, missing_kwds=MISSING)
            g.dissolve().boundary.plot(ax=ax, color="black", linewidth=0.7, zorder=4)
            if overlays and pk in overlays:
                ov = overlays[pk]
                if ov.get("water")    is not None: ov["water"].plot(ax=ax, color="#2c7fb8", linewidth=2, zorder=5)
                if ov.get("poi")      is not None: ov["poi"].plot(ax=ax, color="#222222", markersize=10, zorder=7)
                if ov.get("mrt")      is not None: ov["mrt"].plot(ax=ax, color="#54278f", marker="s", markersize=45, zorder=8)
                if ov.get("entrance") is not None: ov["entrance"].plot(ax=ax, color="#e6550d", marker="^", markersize=60, zorder=8)
            if lisa and pk in lisa and grp == "hdb":
                hot = g[lisa[pk].values]
                if len(hot): hot.boundary.plot(ax=ax, color="black", linewidth=1.4, zorder=6)
            ax.set_aspect("equal"); ax.set_xticks([]); ax.set_yticks([])
            for s in ax.spines.values(): s.set_visible(False)
            if i == 0: ax.set_title(GROUP_LABEL[grp], fontsize=12, fontweight="bold")
            if j == 0:
                ax.set_ylabel(f"{PARK_LABEL[pk]}\n({PARK_TYPE[pk]})", fontsize=10, fontweight="bold")
                add_scalebar(ax, SCALEBAR_M.get(pk, 500))
            if i == 0 and j == 0: add_north(ax)

    fig.subplots_adjust(left=0.075, right=0.97, top=0.90, bottom=0.13, wspace=0.05, hspace=0.12)
    sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm); sm.set_array([])
    cax = fig.add_axes([0.075, 0.06, 0.5, 0.02])
    cb = fig.colorbar(sm, cax=cax, orientation="horizontal")
    cb.solids.set_rasterized(False)
    cb.set_ticks([-2, -1, 0, 1, 2])
    cb.set_ticklabels(["0.25×", "0.5×", "1× (expected)", "2×", "4×"])
    cb.set_label(f"Location quotient — {METRIC_LABEL[metric]}", fontsize=10)
    fig.text(0.075, 0.018, "← under-used by group", fontsize=8, color="#2c7fb8")
    fig.text(0.575, 0.018, "over-used by group →", fontsize=8, color="#b2182b", ha="right")
    handles = [Patch(facecolor="white", edgecolor="#cccccc", label="Unvisited cell (zero ≠ low)")]
    if lisa: handles.append(Patch(facecolor="none", edgecolor="black", linewidth=1.4, label="LISA hot-spot"))
    fig.legend(handles=handles, loc="lower right", bbox_to_anchor=(0.97, 0.04), fontsize=8, frameon=False)
    fig.suptitle(f"Where each housing group over- or under-uses the park — {METRIC_LABEL[metric]}",
                 x=0.075, ha="left", fontsize=15, fontweight="bold")
    if save:
        fig.savefig(save, dpi=220, bbox_inches="tight", facecolor="white"); print("saved", save)
    return fig

In [ ]:
fig_freq = plot_lq_grid(parks, "freq", save=OUTDIR / "fig4a_frequency_LQ.svg")
plt.show()

In [ ]:
fig_dur = plot_lq_grid(parks, "dur", save=OUTDIR / "fig4b_duration_LQ.svg")
plt.show()

In [ ]:
s = summary_table(parks, "all").set_index("park")
labels = [PARK_LABEL[k] + f"\n({PARK_TYPE[k]})" for k in PARK_ORDER]
order  = [PARK_LABEL[k] for k in PARK_ORDER]
s = s.loc[order]
y = np.arange(len(order))[::-1]
colors = ["#3a7d44", "#2c7fb8", "#b2562f"]

fig2, ax = plt.subplots(1, 3, figsize=(13.2, 4.2))
fig2.subplots_adjust(left=0.16, right=0.985, top=0.72, bottom=0.16, wspace=0.42)

# (a) mean vs median frequency -> concentration
a = ax[0]
for k in range(len(order)):
    mean_v, med_v = s["mean_F"].iloc[k], s["median_F"].iloc[k]
    a.plot([med_v, mean_v], [y[k], y[k]], color="#bbbbbb", lw=3, zorder=1, solid_capstyle="round")
    a.scatter(med_v, y[k], s=90, color="#7f7f7f", zorder=3, label="Median" if k == 0 else "")
    a.scatter(mean_v, y[k], s=110, color=colors[k], zorder=3, label="Mean" if k == 0 else "")
    note = f"gap x{mean_v/med_v:.0f}" if med_v > 0 else "median 0"
    a.annotate(note, (max(mean_v, med_v) + 1.5, y[k]), va="center", fontsize=8.5, color="#555555")
a.set_yticks(y); a.set_yticklabels(labels, fontsize=9)
a.set_xlabel("Visits per cell ($F_h$)", fontsize=9.5); a.set_xlim(left=0)
a.set_title("(a) Use is concentrated, not even", fontsize=11, fontweight="bold", loc="left", pad=30)
a.text(0, 1.04, "Large mean-vs-median gap = a few cells carry most visits",
       transform=a.transAxes, fontsize=8.3, style="italic", color="#666666")
a.legend(loc="lower right", fontsize=8.5, frameon=False)
for sp in ["top", "right"]: a.spines[sp].set_visible(False)

# (b) share of area visited
b = ax[1]
share = s["share_visited"].values * 100
b.barh(y, share, color=colors, height=0.55)
for k in range(len(order)):
    b.text(share[k] - 3, y[k], f"{share[k]:.0f}%", va="center", ha="right", color="white", fontweight="bold", fontsize=10)
b.set_yticks(y); b.set_yticklabels([""] * len(order)); b.set_xlim(0, 100)
b.set_xlabel("Share of park cells visited", fontsize=9.5)
b.set_title("(b) Breadth of spatial coverage", fontsize=11, fontweight="bold", loc="left", pad=30)
b.text(0, 1.04, "How much of the park is used at all", transform=b.transAxes, fontsize=8.3, style="italic", color="#666666")
for sp in ["top", "right"]: b.spines[sp].set_visible(False)

# (c) mean aggregate dwell per cell (hours)
c = ax[2]
dwell_h = s["mean_D"].values / 3600.0
c.barh(y, dwell_h, color=colors, height=0.55)
for k in range(len(order)):
    c.text(dwell_h[k] + dwell_h.max() * 0.02, y[k], f"{dwell_h[k]:.0f} h", va="center", fontsize=9.5, color="#333333")
c.set_yticks(y); c.set_yticklabels([""] * len(order)); c.set_xlim(left=0)
c.set_xlabel("Mean aggregate dwell per cell (hours)", fontsize=9.5)
c.set_title("(c) Lingering intensity", fontsize=11, fontweight="bold", loc="left", pad=30)
c.text(0, 1.04, "Aggregate seconds per cell over the whole period", transform=c.transAxes, fontsize=8.3, style="italic", color="#666666")
for sp in ["top", "right"]: c.spines[sp].set_visible(False)

fig2.suptitle("Companion summary — computed from your data", fontsize=12.5, fontweight="bold", x=0.16, ha="left", y=0.965)
fig2.savefig(OUTDIR / "fig4_summary_chart.svg", dpi=220, bbox_inches="tight", facecolor="white")
print("saved", OUTDIR / "fig4_summary_chart.svg")
plt.show()

In [ ]:
try:
    from libpysal.weights import Queen
    from esda.moran import Moran, Moran_Local
    HAVE_PYSAL = True
except Exception as e:
    HAVE_PYSAL = False
    print("Spatial stats skipped - install esda libpysal:", e)

def _queen(g):
    try:
        return Queen.from_dataframe(g, use_index=True)
    except TypeError:
        return Queen.from_dataframe(g)

In [ ]:
def global_moran_table(parks, metric_keys=("freq", "dur"), n_perm=999, seed=42):
    np.random.seed(seed); rows = []
    for pk in PARK_ORDER:
        g = parks[pk]; w = _queen(g); w.transform = "r"
        for m in metric_keys:
            y = np.log1p(g[f"{m}_total"].to_numpy(dtype=float))
            mi = Moran(y, w, permutations=n_perm)
            rows.append(dict(park=pk, metric=m, n=len(g),
                             I=round(mi.I, 4), EI=round(mi.EI, 4),
                             z=round(mi.z_norm, 3), p_sim=round(mi.p_sim, 4)))
    return pd.DataFrame(rows)

if HAVE_PYSAL:
    display(global_moran_table(parks))

In [ ]:
def _lisa_clusters_and_moran(g, metric="freq", n_perm=999, seed=42, alpha=0.05):
    """Return a copied GeoDataFrame with LISA classes plus Moran objects.

    metric must match the existing total columns: freq_total or dur_total.
    """
    np.random.seed(seed)
    out = g.copy()
    w = _queen(out)
    w.transform = "r"
    y = np.log1p(out[f"{metric}_total"].to_numpy(dtype=float))
    mi = Moran(y, w, permutations=n_perm)
    lm = Moran_Local(y, w, permutations=n_perm)
    out["cluster"] = np.where(lm.p_sim < alpha, lm.q, 0)
    out["_z"] = (y - y.mean()) / y.std(ddof=0) if y.std(ddof=0) > 0 else 0
    out["_lag_z"] = w.sparse.dot(out["_z"].to_numpy(dtype=float))
    return out, mi, lm


def _plot_lisa_map(ax, g_lisa, cats):
    """Draw one LISA cluster map on an existing axis."""
    for cval, (lab, col) in cats.items():
        sub = g_lisa[g_lisa["cluster"] == cval]
        if len(sub):
            sub.plot(ax=ax, color=col, edgecolor="white", linewidth=0.2)
    g_lisa.dissolve().boundary.plot(ax=ax, color="black", linewidth=0.7)
    ax.set_aspect("equal")
    ax.set_xticks([])
    ax.set_yticks([])
    for s in ax.spines.values():
        s.set_visible(False)


def plot_lisa(parks, metric="freq", n_perm=999, seed=42, save=None):
    """Original one-metric LISA map, kept for backwards compatibility."""
    np.random.seed(seed)
    cats = {0: ("Not significant", "#e8e8e8"), 1: ("High-High", "#b2182b"),
            2: ("Low-High", "#fdbf9c"), 3: ("Low-Low", "#2166ac"), 4: ("High-Low", "#92c5de")}
    fig, axes = plt.subplots(1, len(PARK_ORDER), figsize=(4.2 * len(PARK_ORDER), 4.4))
    axes = np.atleast_1d(axes)
    for ax, pk in zip(axes, PARK_ORDER):
        g_lisa, mi, _ = _lisa_clusters_and_moran(parks[pk], metric, n_perm=n_perm, seed=seed)
        _plot_lisa_map(ax, g_lisa, cats)
        ax.set_title(f"{PARK_LABEL[pk]}\nGlobal Moran's I = {mi.I:.3f}  (p = {mi.p_sim:.3f})",
                     fontsize=10, fontweight="bold")
    handles = [Patch(facecolor=col, label=lab) for _, (lab, col) in cats.items()]
    fig.legend(handles=handles, loc="lower center", ncol=5, fontsize=9, frameon=False, bbox_to_anchor=(0.5, -0.01))
    fig.suptitle(f"Local spatial clustering (LISA) — {METRIC_LABEL[metric]}", fontsize=14, fontweight="bold")
    fig.tight_layout(rect=[0, 0.05, 1, 0.95])
    if save:
        fig.savefig(save, dpi=220, bbox_inches="tight", facecolor="white")
        print("saved", save)
    return fig


def plot_lisa_integrated(parks, n_perm=999, seed=42, save=None):
    """Three-column LISA figure: Moran summary, frequency clusters, duration clusters.

    Layout follows the reference image:
    - one row per park;
    - left column shows the Moran's I value and a Moran scatter for frequency and duration;
    - middle column shows the LISA map for frequency;
    - right column shows the LISA map for duration.
    """
    cats = {0: ("Not significant", "#eeeeee"), 1: ("High-High", "#b2182b"),
            2: ("Low-High", "#fdbf9c"), 3: ("Low-Low", "#2166ac"), 4: ("High-Low", "#92c5de")}
    metric_label_short = {"freq": "Frequency", "dur": "Duration"}

    # Pre-compute results so stats and maps use the same permutations.
    results = {}
    for pk in PARK_ORDER:
        results[(pk, "freq")] = _lisa_clusters_and_moran(parks[pk], "freq", n_perm=n_perm, seed=seed)
        results[(pk, "dur")] = _lisa_clusters_and_moran(parks[pk], "dur", n_perm=n_perm, seed=seed)

    fig = plt.figure(figsize=(12.2, 10.8))
    gs = fig.add_gridspec(
        nrows=len(PARK_ORDER), ncols=3,
        width_ratios=[1.2, 1.45, 1.45],
        left=0.05, right=0.98, top=0.92, bottom=0.10,
        wspace=0.12, hspace=0.28
    )

    for r, pk in enumerate(PARK_ORDER):
        # Left column: two mini rows for Frequency and Duration.
        sub = gs[r, 0].subgridspec(2, 2, width_ratios=[0.9, 1.4], wspace=0.12, hspace=0.12)
        for mr, metric in enumerate(["freq", "dur"]):
            g_lisa, mi, lm = results[(pk, metric)]

            ax_box = fig.add_subplot(sub[mr, 0])
            p_text = "p<0.001" if mi.p_sim < 0.001 else f"p={mi.p_sim:.3f}"
            sig = "**" if mi.p_sim < 0.01 else ("*" if mi.p_sim < 0.05 else "")
            box_colour = "#8c0026" if mi.p_sim < 0.01 else ("#fddbc7" if mi.I > 0 else "#d1e5f0")
            text_colour = "white" if mi.p_sim < 0.01 else "black"
            ax_box.set_facecolor(box_colour)
            ax_box.text(0.5, 0.5, f"I={mi.I:.3f}{sig}\n{p_text}", ha="center", va="center",
                        fontsize=10, color=text_colour)
            ax_box.set_xticks([]); ax_box.set_yticks([])
            for s in ax_box.spines.values():
                s.set_visible(False)
            ax_box.set_ylabel(metric_label_short[metric], fontsize=10, rotation=90, labelpad=10)

            ax_sc = fig.add_subplot(sub[mr, 1])
            x = g_lisa["_z"].to_numpy(dtype=float)
            y = g_lisa["_lag_z"].to_numpy(dtype=float)
            ax_sc.scatter(x, y, s=7, color="#2b83ba", alpha=0.8, linewidth=0)
            if np.isfinite(x).all() and np.isfinite(y).all() and len(x) > 1:
                b1, b0 = np.polyfit(x, y, 1)
                xx = np.linspace(np.nanmin(x), np.nanmax(x), 100)
                ax_sc.plot(xx, b1 * xx + b0, color="#b2182b", lw=1.4)
            ax_sc.axhline(0, color="#999999", lw=0.6, ls=":")
            ax_sc.axvline(0, color="#999999", lw=0.6, ls=":")
            ax_sc.tick_params(labelsize=6, length=2)
            ax_sc.set_xlabel(f"Standardized log({metric_label_short[metric].lower()} + 1)", fontsize=6)
            ax_sc.set_ylabel("Spatial lag", fontsize=6)
            for s in ["top", "right"]:
                ax_sc.spines[s].set_visible(False)

        # Middle and right columns: LISA maps.
        for c, metric in enumerate(["freq", "dur"], start=1):
            ax_map = fig.add_subplot(gs[r, c])
            g_lisa, mi, _ = results[(pk, metric)]
            _plot_lisa_map(ax_map, g_lisa, cats)
            add_scalebar(ax_map, SCALEBAR_M.get(pk, 500))
            if r == 0:
                ax_map.set_title(metric_label_short[metric], fontsize=14, fontweight="bold", pad=10)

        # Park label on the left of each row.
        fig.text(0.055, 0.865 - r * 0.282, PARK_LABEL[pk], fontsize=14, ha="left", va="center")

    handles = [Patch(facecolor=col, edgecolor="none", label=lab) for _, (lab, col) in cats.items()]
    fig.legend(handles=handles, loc="lower center", ncol=5, fontsize=9, frameon=False, bbox_to_anchor=(0.56, 0.035))
    fig.suptitle("Local spatial clustering (LISA): frequency and duration", fontsize=16, fontweight="bold", x=0.48)

    if save:
        fig.savefig(save, dpi=220, bbox_inches="tight", facecolor="white")
        print("saved", save)
    return fig


if HAVE_PYSAL:
    # Keep the original frequency output.
    plot_lisa(parks, "freq", save=OUTDIR / "fig_lisa_frequency.svg")
    plt.show()

    # New separate duration output.
    plot_lisa(parks, "dur", save=OUTDIR / "fig_lisa_duration.svg")
    plt.show()

    # New integrated three-column figure, matching the reference layout.
    plot_lisa_integrated(parks, save=OUTDIR / "fig_lisa_frequency_duration_integrated.svg")
    plt.show()


In [ ]:
# Added for the public release. The manuscript reports p_sim from the per-call path above
# (_lisa_clusters_and_moran re-seeds 42 for each park x outcome), not the seed-once global_moran_table.
rows = []
for pk in PARK_ORDER:
    for m in ("freq", "dur"):
        _, mi, _ = _lisa_clusters_and_moran(parks[pk], m)
        rows.append(dict(park=pk, outcome=m, n_hex=len(parks[pk]), weights="Queen contiguity",
                         moran_I=mi.I, expected_I=mi.EI, z_norm=mi.z_norm, p_sim=mi.p_sim, permutations=999, seed=42))
manuscript_moran = pd.DataFrame(rows)
manuscript_moran.to_csv(OUTDIR / "full_grid_moran_summary.csv", index=False)
display(manuscript_moran)


In [ ]:
# Example (uncomment and point to your files):
# overlays = {
#     "amk": {
#         "poi":      gpd.read_file("amk_services.geojson").to_crs(PROJ_CRS),
#         "mrt":      gpd.read_file("mrt_stations.geojson").to_crs(PROJ_CRS),
#         "water":    gpd.read_file("kallang_river.geojson").to_crs(PROJ_CRS),
#         "entrance": None,
#     },
#     # "jurong": {...}, "punggol": {...}
# }
# plot_lq_grid(parks, "freq", overlays=overlays, save=OUTDIR / "fig4a_frequency_LQ_overlays.svg")

In [ ]:
diag = []
for pk in PARK_ORDER:
    g = parks[pk]; vis = g["freq_total"] > 0
    f = g.loc[vis, "freq_total"]; d = g.loc[vis, "dur_total"]
    diag.append(dict(
        park=PARK_LABEL[pk], n_visited=int(vis.sum()),
        r_total=round(float(d.corr(f)), 3),
        r_loglog=round(float(np.log1p(d).corr(np.log1p(f))), 3),
    ))
print("Pearson correlation of per-hex duration vs frequency (high => redundant):")
display(pd.DataFrame(diag))

In [ ]:
# log-log scatter; colour = mean stay per visit (the orthogonal dimension)
fig, axes = plt.subplots(1, len(PARK_ORDER), figsize=(4.3 * len(PARK_ORDER), 3.9))
axes = np.atleast_1d(axes)
for a, pk in zip(axes, PARK_ORDER):
    g = parks[pk]; vis = (g["freq_total"] > 0) & (g["dur_total"] > 0)
    f  = g.loc[vis, "freq_total"].to_numpy(float)
    dh = g.loc[vis, "dur_total"].to_numpy(float) / 3600.0           # hours
    ms = (g.loc[vis, "dur_total"] / g.loc[vis, "freq_total"] / 60.0).to_numpy()  # min/visit
    vmax = np.nanpercentile(ms, 95) if ms.size else 1.0
    sc = a.scatter(f, dh, c=ms, cmap="viridis", s=14, alpha=0.8,
                   norm=mcolors.Normalize(0, max(vmax, 1.0)))
    a.set_xscale("log"); a.set_yscale("log")
    a.set_xlabel("visits per cell ($F_h$)", fontsize=9)
    a.set_ylabel("aggregate dwell per cell (h)", fontsize=9)
    a.set_title(PARK_LABEL[pk], fontsize=10, fontweight="bold")
    cb = fig.colorbar(sc, ax=a, fraction=0.046, pad=0.04); cb.solids.set_rasterized(False); cb.set_label("min / visit", fontsize=8)
    for sp in ["top", "right"]: a.spines[sp].set_visible(False)
fig.suptitle("Frequency vs duration: spread off the diagonal = where mean stay differs",
             fontsize=12, fontweight="bold")
fig.tight_layout()
fig.savefig(OUTDIR / "fig_freq_vs_dur_diagnostic.svg", dpi=200, bbox_inches="tight", facecolor="white")
plt.show()

In [ ]:
def _draw_grid(value_of, cmap, norm, suptitle, cbar_label, left_lab, right_lab,
               missing_lab, cbar_ticks=None, cbar_ticklabels=None, save=None):
    # generic parks x groups grid; value_of(pk, grp, gdf) returns a Series to colour
    cmap = plt.get_cmap(cmap) if isinstance(cmap, str) else cmap
    nr, nc = len(PARK_ORDER), len(GROUPS)
    fig, axes = plt.subplots(nr, nc, figsize=(4.0 * nc, 3.7 * nr))
    for i, pk in enumerate(PARK_ORDER):
        for j, grp in enumerate(GROUPS):
            ax = axes[i, j]; g = parks[pk].copy()
            g["_v"] = value_of(pk, grp, g)
            g.plot(column="_v", cmap=cmap, norm=norm, ax=ax, edgecolor="white",
                   linewidth=0.2, missing_kwds=MISSING)
            g.dissolve().boundary.plot(ax=ax, color="black", linewidth=0.7, zorder=4)
            ax.set_aspect("equal"); ax.set_xticks([]); ax.set_yticks([])
            for s in ax.spines.values(): s.set_visible(False)
            if i == 0: ax.set_title(GROUP_LABEL[grp], fontsize=12, fontweight="bold")
            if j == 0:
                ax.set_ylabel(f"{PARK_LABEL[pk]}\n({PARK_TYPE[pk]})", fontsize=10, fontweight="bold")
                add_scalebar(ax, SCALEBAR_M.get(pk, 500))
            if i == 0 and j == 0: add_north(ax)
    fig.subplots_adjust(left=0.075, right=0.97, top=0.90, bottom=0.13, wspace=0.05, hspace=0.12)
    sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm); sm.set_array([])
    cax = fig.add_axes([0.075, 0.06, 0.5, 0.02])
    cb = fig.colorbar(sm, cax=cax, orientation="horizontal")
    cb.solids.set_rasterized(False)
    if cbar_ticks is not None:
        cb.set_ticks(cbar_ticks)
        if cbar_ticklabels is not None: cb.set_ticklabels(cbar_ticklabels)
    cb.set_label(cbar_label, fontsize=10)
    if left_lab:  fig.text(0.075, 0.018, left_lab, fontsize=8, color="#2c7fb8")
    if right_lab: fig.text(0.575, 0.018, right_lab, fontsize=8, color="#b2182b", ha="right")
    fig.legend(handles=[Patch(facecolor="white", edgecolor="#cccccc", label=missing_lab)],
               loc="lower right", bbox_to_anchor=(0.97, 0.04), fontsize=8, frameon=False)
    fig.suptitle(suptitle, x=0.075, ha="left", fontsize=15, fontweight="bold")
    if save:
        fig.savefig(save, dpi=220, bbox_inches="tight", facecolor="white"); print("saved", save)
    return fig

In [ ]:
def plot_difference_map(parks, min_visits=3, save=None):
    # log2(LQ_dur / LQ_freq): red = lingers (dwell share > visit share); blue = passes through
    series, vals = {}, []
    for pk in PARK_ORDER:
        g = parks[pk]
        for grp in GROUPS:
            f = g[f"{PREFIX['freq']}{grp}"]
            ratio = g[f"LQ_dur_{grp}"] / g[f"LQ_freq_{grp}"]
            lr = np.log2(ratio.where((ratio > 0) & np.isfinite(ratio)))
            lr = lr.where(f >= min_visits)            # reliability floor
            series[(pk, grp)] = lr
            vals.append(lr.dropna().to_numpy())
    vals = np.concatenate(vals) if vals else np.array([0.0])
    vmax = np.nanpercentile(np.abs(vals), 95) if vals.size else 1.0
    vmax = max(0.5, np.ceil(vmax * 2) / 2)            # tidy symmetric limit, >= 0.5
    norm = mcolors.Normalize(-vmax, vmax)
    ticks = list(np.arange(-vmax, vmax + 1e-9, vmax / 2))
    ticklabels = [f"{2.0**t:.2g}×" for t in ticks]
    def value_of(pk, grp, g): return series[(pk, grp)].clip(-vmax, vmax)
    return _draw_grid(value_of, "RdBu_r", norm,
        "Do they linger or pass through?  Dwell LQ vs visit LQ, by housing group",
        "dwell LQ ÷ visit LQ  (log₂ ratio)",
        "← passes through (visits > stays)", "lingers (stays > visits) →",
        f"No data / < {min_visits} visits", cbar_ticks=ticks, cbar_ticklabels=ticklabels,
        save=save)

fig_diff = plot_difference_map(parks, min_visits=3, save=OUTDIR / "fig4c_dwell_vs_visit.svg")
plt.show()

In [ ]:
def plot_meanstay_map(parks, min_visits=3, cmap="viridis", save=None):
    # mean stay per visit (minutes) = D_h_<g> / F_h_<g>; sequential scale
    series, vals = {}, []
    for pk in PARK_ORDER:
        g = parks[pk]
        for grp in GROUPS:
            f = g[f"{PREFIX['freq']}{grp}"].astype(float)
            d = g[f"{PREFIX['dur']}{grp}"].astype(float)
            ms = (d / f) / 60.0
            ms = ms.where(f >= min_visits)            # reliability + avoids /0
            series[(pk, grp)] = ms
            vals.append(ms.dropna().to_numpy())
    vals = np.concatenate(vals) if vals else np.array([0.0, 1.0])
    vmax = np.nanpercentile(vals, 95) if vals.size else 1.0
    vmax = max(vmax, 1.0)
    norm = mcolors.Normalize(0, vmax)
    def value_of(pk, grp, g): return series[(pk, grp)].clip(0, vmax)
    return _draw_grid(value_of, cmap, norm,
        "How long do they stay per visit?  Mean dwell per visit ($D_h / F_h$)",
        f"mean stay per visit (minutes, capped at {vmax:.0f})",
        None, None, f"No data / < {min_visits} visits", save=save)

fig_ms = plot_meanstay_map(parks, min_visits=3, save=OUTDIR / "fig4d_mean_stay.svg")
plt.show()